# 03 Define Support Mask

Loads the FTH HDF5 result and defines the real-space support mask. Choose **one** method: Paint PNG, Napari, or circular support coordinates. Every method produces the same binary `supportmask`, which is saved as both PNG and in the HDF5 file.

In [18]:
# Configure Qt before importing pyplot or any other GUI-aware library.
import os
import sys
from os.path import join

BASEFOLDER = os.path.abspath(os.getcwd())
LIBRARY_FOLDER = join(BASEFOLDER, "library")
if LIBRARY_FOLDER not in sys.path:
    sys.path.insert(0, LIBRARY_FOLDER)

from notebook_setup import configure_matplotlib_qt
MATPLOTLIB_BACKEND = configure_matplotlib_qt()
print("Matplotlib backend:", MATPLOTLIB_BACKEND)

import os, sys
from os.path import join
from glob import glob
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
# pyFAI is not required by this notebook.


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
# PETRA loading helpers are not required by this SOLEIL workflow.
import fth_phase_workflow as wf

wf = reload(wf)  # Refresh helpers when rerunning in an existing kernel.
import painted_masks as pm

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR

    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci

    PhR = None
    GPU = False
    print("GPU unavailable")

try:
    %load_ext jupyter_black
except Exception:
    pass

Matplotlib backend: qt5 (pyqt5)
Base folder: /home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS
GPU available


In [19]:
im_ids: {
    "+": [110, 113, 115, 117, 119, 121, 123],
    "-": [111, 114, 116, 118, 120, 122, 124],
}
dark_ids: {"+": 112, "-": 112}

im_ids: {'+': array([613]), '-': [614]}
dark_ids: {'+': np.int64(604), '-': 604}

NameError: name 'array' is not defined

In [20]:
# Choose the positive-helicity image ID whose support mask you want to define.
im_id = 613
# True applies the centered mask_pixel created in notebook 00/00a.
# The support preview is reconstructed from the filtered hologram when enabled.
USE_MASK_PIXEL = True

BASEFOLDER = find_basefolder()
data_h5_matches = glob(
    join(BASEFOLDER, "processed", "Logs", f"data_recon_ImId_{im_id:04d}_*.hdf5")
)
if not data_h5_matches:
    raise FileNotFoundError(f"No reconstruction HDF5 found for im_id={im_id}.")
if len(data_h5_matches) > 1:
    raise RuntimeError(
        f"Multiple reconstruction HDF5 files found for im_id={im_id}: {data_h5_matches}"
    )
DATA_H5 = data_h5_matches[0]
data = wf.load_data_dict(DATA_H5)
positive_label = data["positive_label"]
reference_label = data["reference_label"]
experimental_setup = data["experimental_setup"]
focus_fth = data.get("focus_fth", {})
prop_dist = focus_fth.get("prop_dist", 0)
phase = focus_fth.get("phase", 0)
beamstop_recipe = dict(
    data.get("mask_beamstop_smooth_recipe")
    or data.get("mask_pixel_smooth_recipe", {"radius": 35, "order": 4})
)
shape = data["holo"][positive_label]["image_c"].shape
mask_beamstop_smooth = wf.butterworth_disk_mask(
    shape, beamstop_recipe["radius"], beamstop_recipe["order"]
)
stored_mask_pixel = np.asarray(data.get("mask_pixel", np.zeros(shape)), dtype=np.uint8)
if stored_mask_pixel.shape != shape:
    raise ValueError(
        f"mask_pixel shape {stored_mask_pixel.shape} != hologram shape {shape}"
    )
mask_pixel = stored_mask_pixel if USE_MASK_PIXEL else np.zeros(shape, dtype=np.uint8)
mask_pixel_fth_recipe = dict(
    data.get("mask_pixel_fth_recipe", {"dilation_pixels": 3, "sigma": 3})
)
# Gaussian filtering must operate on floating-point data, not uint8 labels.
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float),
    mask_pixel_fth_recipe["dilation_pixels"],
    mask_pixel_fth_recipe["sigma"],
)
print(f"mask_pixel filtering: {'enabled' if USE_MASK_PIXEL else 'disabled'}")
print("Loaded:", DATA_H5)

mask_pixel filtering: enabled
Loaded: /home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/processed/Logs/data_recon_ImId_0613_rb.hdf5


## Build support-mask preview reconstruction

In [21]:
pos = np.asarray(data["holo"][positive_label]["image_c"], dtype=float) / data["factor"]
neg = np.asarray(data["holo"][reference_label]["image_c"], dtype=float)
holo_unmasked = pos + neg - data["offset"]
# This support image is filtered in detector/Fourier space before reconstruction.
holo = holo_unmasked * (1 - mask_beamstop_smooth) * (1 - mask_pixel_fth)
recon = wf.fth_reconstruct(
    holo,
    experimental_setup,
    fth,
    prop_dist=prop_dist,
    phase=phase,
)
recon = np.abs(recon)
recon_support = recon  # Backward-compatible name used by the mask widgets below.
# cimshow(recon)

## Paint support mask with PNG

In [22]:
# Option: export the reconstructed amplitude, paint support pixels bright red,
# save the edited image under the printed mask filename, then run the next cell.
support_png, support_painted_png = pm.mask_png_paths(
    BASEFOLDER,
    "supportmask",
    data["holo"][positive_label]["id"],
)
pm.save_mask_reference_png(
    recon_support,
    support_png,
    log_scale=True,
)
print("Paint bright red support pixels in:")
print(support_png)
print("Save the edited PNG as:")
print(support_painted_png)

Paint bright red support pixels in:
/home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/processed/supportmask/supportmask_[613]_reference.png
Save the edited PNG as:
/home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/processed/supportmask/supportmask_[613].png


In [23]:
# Option: load bright-red pixels from the edited PNG as supportmask.
supportmask = pm.load_bright_red_mask_png(
    support_painted_png,
    expected_shape=recon_support.shape,
)
support_coordinates = []
sample = "paint_png"

fig, ax = plt.subplots(figsize=(6, 6))
vmin, vmax = np.percentile(recon_support, (1, 99))
ax.imshow(recon_support, vmin=vmin, vmax=vmax, cmap="gray")
ax.imshow(supportmask, alpha=0.4, cmap="binary")
ax.set_title("painted supportmask overlay")

FileNotFoundError: [Errno 2] No such file or directory: '/home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/processed/supportmask/supportmask_[613].png'

## Napari support mask (alternative to Paint)

Run these two cells instead of the Paint or coordinate sections. Paint label 1 over the allowed real-space support; label 0 erases.

In [ ]:
try:
    import napari
    from napari.utils.colormaps import DirectLabelColormap
except ImportError as exc:
    raise ImportError(
        "Install napari[all] to use this optional support-mask method."
    ) from exc
%gui qt

initial_support = np.asarray(
    data.get("supportmask", np.zeros(recon_support.shape)), dtype=np.uint8
)
if initial_support.shape != recon_support.shape:
    initial_support = np.zeros(recon_support.shape, dtype=np.uint8)
try:
    support_viewer.close()
except NameError:
    pass
support_viewer = napari.Viewer(title="supportmask")
support_viewer.add_image(recon_support, name="FTH amplitude", colormap="gray")
support_layer = support_viewer.add_labels(
    initial_support,
    name="supportmask",
    opacity=0.5,
    colormap=DirectLabelColormap(
        color_dict={
            0: np.array([0.0, 0.0, 0.0, 0.0]),
            1: np.array([1.0, 0.0, 0.0, 1.0]),
            None: np.array([1.0, 0.0, 0.0, 1.0]),
        }
    ),
)
support_layer.selected_label = 1
support_viewer.layers.selection.active = support_layer
print("Paint label 1 for allowed support and label 0 to erase.")
print("When finished, run the next cell.")

In [24]:
supportmask = (np.asarray(support_layer.data) > 0).astype(np.uint8)
if supportmask.shape != recon_support.shape:
    raise ValueError(
        f"Napari support has shape {supportmask.shape}, expected {recon_support.shape}"
    )
support_coordinates = []
sample = "napari"
print(f"Napari support contains {int(supportmask.sum())} pixels.")

NameError: name 'support_layer' is not defined

## Support coordinates / circle widget (alternative)

Run this section instead of Paint or Napari. Supply `(y, x, radius)` circles directly, or adjust them with the widget before creating the mask.

In [25]:
plt.close("all")
def get_supportmask_coordinates(sample):
    """
    Dictionary that stores coordinates of circular support mask apertures.
    Taken from FTH_CDI_01.ipynb.
    """
    off = 0
    support_coord = dict()
    support_coord["kri2"] = [
        (1024 + off, 1024 + off, 14.0),
        (689.50 + off, 620 + off, 83.0),
    ]
    support_coord["kri2"] = [
        (1024 + off, 1024 + off, 14.0),
        (689.50 + off, 620 + off, 83.0),
    ]
    support_coord["kri2"] = [
        (1024 + off, 1024 + off, 14.0),
        (457.50 + off, 346.5 + off, 127.5),
    ]
    return support_coord[sample]


# Select stored coordinates, or replace this with your own list of
# (y, x, radius) tuples. Set False to use them directly without a widget.
sample = "kri2"
support_coordinates = get_supportmask_coordinates(sample)
ADJUST_COORDINATES_WITH_WIDGET = True

if ADJUST_COORDINATES_WITH_WIDGET:
    print("Cover the object and reference apertures with circles.")
    ds_circle = interactive.InteractiveCircleCoordinates(
        recon_support,
        len(support_coordinates),
        coordinates=support_coordinates,
    )
else:
    print("Using support_coordinates directly without the widget.")

Cover the object and reference apertures with circles.
Use circle index slider to change between circles. The active circle is highlighted in red.
Right click to move circle to mouse position!


interactive(children=(FloatRangeSlider(value=(2138.205120775711, 51592417.63391894), description='contrast', l…

interactive(children=(IntSlider(value=0, description='index', max=1), Output()), _dom_classes=('widget-interac…

interactive(children=(FloatSlider(value=14.0, description='radius', layout=Layout(width='350px'), max=400.0, s…

In [26]:
if ADJUST_COORDINATES_WITH_WIDGET:
    support_coordinates = ds_circle.get_params()
supportmask = mask_lib.create_circle_supportmask(
    support_coordinates, recon_support.shape
)
supportmask = (supportmask > 0).astype(np.uint8)

fig, ax = plt.subplots(figsize=(6, 6))
vmin, vmax = np.percentile(recon_support, (1, 99))
ax.imshow(recon_support, vmin=vmin, vmax=vmax, cmap="gray")
ax.imshow(supportmask, alpha=0.4, cmap="binary")
ax.set_title("supportmask overlay")

Text(0.5, 1.0, 'supportmask overlay')

## Define CDI ROI

In [27]:
# Zoom/pan to the useful reconstruction area, then execute the next cell.
fig, ax = cimshow(supportmask.astype(int))

interactive(children=(FloatRangeSlider(value=(0.0, 1.0), description='contrast', layout=Layout(width='500px'),…

In [28]:
roi_cdi_s = interactive.axis_to_roi(ax)
roi_cdi = wf.slices_to_roi(roi_cdi_s)
print("CDI ROI:", roi_cdi)

CDI ROI: [0, 2048, 0, 2048]


## Save support mask

In [29]:
im_ids = data["holo"][positive_label]["id"]
im_id = im_ids[0] if isinstance(im_ids, (list, tuple, np.ndarray)) else im_ids
supportmask_png = pm.save_binary_mask_png(BASEFOLDER, "supportmask", im_id, supportmask)
for key in [
    "dark_id_im",
    "dark_id_topo",
    "im_id",
    "topo_id",
    "fth_hologram",
    "fth_hologram_unmasked",
    "fth_png_title",
    "fth_recon",
    "fth_recon_unmasked",
    "mask_pixel_smooth",
    "mask_multiplier",
    "sum_c",
    "diff_c",
    "mask_pixel_c",
    "mask_pixel_c_png",
    "prop_dist",
    "phase",
    "dx",
    "dy",
    "focus_operation",
    "roi",
    "recon_cdi",
    "recon_topo_cdi",
    "phase_retrieval_png",
    "roi_cdi",
    "retrieved_type",
    "phase_cdi",
    "prop_dist_cdi",
    "dx_cdi",
    "dy_cdi",
    "focus_mode_cdi",
    "roi_crop",
    "mask_bs_cdi",
]:
    data.pop(key, None)
data.update(
    {
        "supportmask": supportmask,
        "support_coordinates": np.asarray(support_coordinates, dtype=float),
        "support_sample": sample,
        "supportmask_png": str(supportmask_png),
    }
)
wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Updated supportmask in:", DATA_H5)
print("Saved supportmask PNG:", supportmask_png)

Updated supportmask in: /home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/processed/Logs/data_recon_ImId_0613_rb.hdf5
Saved supportmask PNG: /home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/processed/supportmask/supportmask_613.png


In [30]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_ids:", _summary_im)
print("topo_ids:", _summary_topo)
print("dark_ids (+):", _summary_holo.get(_summary_pos, {}).get("dark_id"))
print("dark_ids (-):", _summary_holo.get(_summary_ref, {}).get("dark_id"))
print("HDF5:", _summary_h5)

im_ids: [613]
topo_ids: [614]
dark_ids (+): 604
dark_ids (-): 604
HDF5: /home/riccardo/Desktop/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS/processed/Logs/data_recon_ImId_0613_rb.hdf5


In [31]:
plt.close("all")

In [32]:
# Acquisition ID summary
_id_holo = data.get("holo", {})
print("im_ids:", {label: state.get("id") for label, state in _id_holo.items()})
print("dark_ids:", {label: state.get("dark_id") for label, state in _id_holo.items()})

im_ids: {'+': array([613]), '-': [614]}
dark_ids: {'+': 604, '-': 604}
